# TARDIS — Step 1 : Data Exploration & Cleaning

Ce notebook charge `dataset.csv`, le nettoie, cree de nouvelles variables et
enregistre le resultat dans `cleaned_dataset.csv`.

## 1. Imports, chargement et inspection du dataset

`pandas` sert a manipuler le tableau de donnees, `numpy` fournit les bornes
infinies utilisees plus loin pour decouper les valeurs en categories.

`pd.read_csv()` lit un fichier CSV et le transforme en tableau, appele DataFrame.

Deux arguments comptent ici :

- `sep=";"` parce que dans ce fichier les colonnes sont separees par des
  points-virgules et non par des virgules
- `dtype=str` force le chargement de tout le fichier en texte. On veut voir les
  valeurs telles qu'elles sont ecrites avant de convertir quoi que ce soit, sinon
  pandas devine les types et detruit silencieusement les valeurs mal formatees

`.shape` donne la taille du tableau sous la forme (nombre de lignes, nombre de colonnes).

In [22]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/dataset.csv", sep=";", dtype=str)

print("Taille du dataset :", df.shape)

Taille du dataset : (12070, 26)


### 1.1 Apercu des premieres lignes

`.head()` affiche les 5 premieres lignes, pour voir a quoi ressemblent les donnees.

In [23]:
df.head()

,Date,Service,Departure station,Arrival station,Average journey time,Number of scheduled trains,Number of cancelled trains,Cancellation comments,Number of trains delayed at departure,Average delay of late trains at departure,...,Number of trains delayed > 15min,Average delay of trains > 15min (if competing with flights),Number of trains delayed > 30min,Number of trains delayed > 60min,Pct delay due to external causes,Pct delay due to infrastructure,Pct delay due to traffic management,Pct delay due to rolling stock,Pct delay due to station management and equipment reuse,"Pct delay due to passenger handling (crowding, disabled persons, connections)"
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141.0,870,5.0,NaN,289.0,11.24780854,...,110.0,6.51,44.0,8.0,36.13445378,31.09243697,10.92436975,15.96638655,"5,04",0.840336134
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56.0,406.0,1.0,NaN,213.0,8.479968701,...,32.0,5.363539095,9.0,4.0,20.0,35.0,16.66666667,16.66666667,8.333333333,3.333333333
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166.0,226.0,0.0,NaN,21.0,6.23968254,...,11.0,2.938053097,6.0,1.0,22.22222222,27.77777778,16.66666667,16.66666667,5.555555556,11.11111111
3,2018-01,National,PARIS MONTPARNASSE,NANTES,216.21,508.0,3.0,NaN,71.0,7.235211268,...,39.0,5.292211221,18.0,NaN,33.33333333,22.22222222,16.66666667,20.37037037,5.555555556,1.851851852
4,2018-01,National,POITIERS,PARIS MONTPARNASSE,94.0,472.0,4.0,NaN,224.0,6.784672619,...,42.0,4.882371795,10.0,0.0,15.78947368,45.61403509,NaN,15.78947368,1.754385965,1.754385965


### 1.2 Structure et types des colonnes

`.info()` affiche la liste des colonnes, leur type et le nombre de valeurs remplies.

In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12070 entries, 0 to 12069
Data columns (total 26 columns):
 #   Column                                                                         Non-Null Count  Dtype
---  ------                                                                         --------------  -----
 0   Date                                                                           12010 non-null  str  
 1   Service                                                                        11830 non-null  str  
 2   Departure station                                                              12011 non-null  str  
 3   Arrival station                                                                12011 non-null  str  
 4   Average journey time                                                           11830 non-null  str  
 5   Number of scheduled trains                                                     11830 non-null  str  
 6   Number of cancelled trains                       

### 1.3 Comptage des valeurs manquantes

`.isna()` met `True` partout ou la valeur est manquante, `.sum()` additionne ces
`True` et donne donc le nombre de valeurs manquantes de chaque colonne.

In [25]:
print(df.isna().sum())

Date                                                                                60
Service                                                                            240
Departure station                                                                   59
Arrival station                                                                     59
Average journey time                                                               240
Number of scheduled trains                                                         240
Number of cancelled trains                                                         239
Cancellation comments                                                            11493
Number of trains delayed at departure                                              240
Average delay of late trains at departure                                          239
Average delay of all trains at departure                                           241
Departure delay comments                   

### 1.4 Comptage des lignes en double

`.duplicated()` met `True` sur chaque ligne qui existe deja plus haut dans le
tableau, `.sum()` compte ces `True` et donne le nombre de lignes en double.

In [26]:
print("Nombre de lignes en double :", df.duplicated().sum())

Nombre de lignes en double : 174


### 1.5 Bilan des problemes detectes

L'inspection revele que le fichier a ete volontairement sali :

- des valeurs manquantes dans presque toutes les colonnes
- 174 lignes en double
- toutes les colonnes sont du texte, meme celles qui contiennent des nombres
- les nombres sont pollues : `5,04`, `163.0 min`, `16.67%`, des espaces en trop
- la colonne Date a cinq formats : `2018-01`, `2018/01`, `2018 01`, `01-2018`, `2018-01 `
- les gares sont ecrites de plusieurs facons : `LE MANS`, `le mans`, ` LE MANS `

Le reste du notebook corrige ces problemes dans l'ordre.

## 2. Renommage des colonnes

Les noms d'origine sont longs et contiennent des espaces, des parentheses et le
signe `>`. Impossible d'ecrire `df.Average journey time`.

On construit un dictionnaire, ancien nom a gauche et nouveau nom a droite, puis
`.rename(columns=...)` remplace les noms en suivant ce dictionnaire.

`.columns` donne la liste des noms de colonnes, `list()` l'affiche proprement.

In [27]:
nouveaux_noms = {
    "Date": "date",
    "Service": "service",
    "Departure station": "departure_station",
    "Arrival station": "arrival_station",
    "Average journey time": "journey_time",
    "Number of scheduled trains": "nb_scheduled",
    "Number of cancelled trains": "nb_cancelled",
    "Cancellation comments": "cancel_comment",
    "Number of trains delayed at departure": "nb_delayed_dep",
    "Average delay of late trains at departure": "avg_delay_late_dep",
    "Average delay of all trains at departure": "avg_delay_all_dep",
    "Departure delay comments": "dep_comment",
    "Number of trains delayed at arrival": "nb_delayed_arr",
    "Average delay of late trains at arrival": "avg_delay_late_arr",
    "Average delay of all trains at arrival": "avg_delay_all_arr",
    "Arrival delay comments": "arr_comment",
    "Number of trains delayed > 15min": "nb_delayed_15",
    "Average delay of trains > 15min (if competing with flights)": "avg_delay_15",
    "Number of trains delayed > 30min": "nb_delayed_30",
    "Number of trains delayed > 60min": "nb_delayed_60",
    "Pct delay due to external causes": "pct_cause_external",
    "Pct delay due to infrastructure": "pct_cause_infra",
    "Pct delay due to traffic management": "pct_cause_traffic",
    "Pct delay due to rolling stock": "pct_cause_rolling_stock",
    "Pct delay due to station management and equipment reuse": "pct_cause_station",
    "Pct delay due to passenger handling (crowding, disabled persons, connections)": "pct_cause_passenger",
}

df = df.rename(columns=nouveaux_noms)

print(list(df.columns))

['date', 'service', 'departure_station', 'arrival_station', 'journey_time', 'nb_scheduled', 'nb_cancelled', 'cancel_comment', 'nb_delayed_dep', 'avg_delay_late_dep', 'avg_delay_all_dep', 'dep_comment', 'nb_delayed_arr', 'avg_delay_late_arr', 'avg_delay_all_arr', 'arr_comment', 'nb_delayed_15', 'avg_delay_15', 'nb_delayed_30', 'nb_delayed_60', 'pct_cause_external', 'pct_cause_infra', 'pct_cause_traffic', 'pct_cause_rolling_stock', 'pct_cause_station', 'pct_cause_passenger']


## 3. Nettoyage des colonnes texte

La meme gare est ecrite de plusieurs facons. Tant qu'on ne les uniformise pas,
` LYON PART DIEU ` et `LYON PART DIEU` sont deux gares differentes pour pandas,
donc deux categories differentes pour le modele et deux entrees dans le menu
deroulant du dashboard.

Les methodes appliquees :

- `.unique()` donne la liste des valeurs differentes, `len()` compte combien il y en a
- `.str.strip()` enleve les espaces au debut et a la fin du texte
- `.str.upper()` met tout en majuscules, pour que `le mans` et `LE MANS` deviennent identiques
- `.str.replace("SAINT", "ST")` regle le dernier cas : `MARSEILLE SAINT CHARLES` et
  `MARSEILLE ST CHARLES` sont la meme gare

Le compteur avant et apres mesure l'effet du nettoyage : 133 gares distinctes
avant, 61 apres.

In [28]:
print("Gares differentes avant :", len(df["departure_station"].unique()))

df["departure_station"] = df["departure_station"].str.strip().str.upper()
df["arrival_station"] = df["arrival_station"].str.strip().str.upper()
df["service"] = df["service"].str.strip().str.upper()

df["departure_station"] = df["departure_station"].str.replace(
    "SAINT", "ST", regex=False
)
df["arrival_station"] = df["arrival_station"].str.replace("SAINT", "ST", regex=False)

print("Gares differentes apres :", len(df["departure_station"].unique()))

Gares differentes avant : 133
Gares differentes apres : 61


## 4. Conversion des colonnes en nombres

Dix-neuf colonnes doivent contenir des nombres mais sont chargees en texte, avec
quatre defauts differents. Une boucle `for` applique le meme traitement a chacune.

Ordre des operations, chaque etape reglant un defaut observe :

- `.str.strip()` enleve les espaces autour de la valeur : `" 6.51 "` devient `"6.51"`
- `.str.replace(" min", "")` enleve l'unite : `"163.0 min"` devient `"163.0"`
- `.str.replace("%", "")` enleve le symbole : `"16.67%"` devient `"16.67"`
- `.str.replace(",", ".")` corrige la decimale francaise : `"5,04"` devient `"5.04"`
- `pd.to_numeric()` transforme enfin le texte en nombre

L'argument `errors="coerce"` fait passer a `NaN`, c'est-a-dire vide, ce qui reste
inconvertible, au lieu de faire planter le notebook.

L'ordre compte. Si `pd.to_numeric` passait avant les `.replace`, il transformerait
`"5,04"` en `NaN` et le nettoyage arriverait trop tard, sur une valeur deja
detruite.

`.dtypes` affiche le type de chaque colonne, on verifie qu'on obtient bien des
`float64`.

In [29]:
colonnes_nombres = [
    "journey_time",
    "nb_scheduled",
    "nb_cancelled",
    "nb_delayed_dep",
    "avg_delay_late_dep",
    "avg_delay_all_dep",
    "nb_delayed_arr",
    "avg_delay_late_arr",
    "avg_delay_all_arr",
    "nb_delayed_15",
    "avg_delay_15",
    "nb_delayed_30",
    "nb_delayed_60",
    "pct_cause_external",
    "pct_cause_infra",
    "pct_cause_traffic",
    "pct_cause_rolling_stock",
    "pct_cause_station",
    "pct_cause_passenger",
]

for col in colonnes_nombres:
    df[col] = df[col].str.strip()
    df[col] = df[col].str.replace(" min", "", regex=False)
    df[col] = df[col].str.replace("%", "", regex=False)
    df[col] = df[col].str.replace(",", ".", regex=False)
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[colonnes_nombres].dtypes)

journey_time               float64
nb_scheduled               float64
nb_cancelled               float64
nb_delayed_dep             float64
avg_delay_late_dep         float64
avg_delay_all_dep          float64
nb_delayed_arr             float64
avg_delay_late_arr         float64
avg_delay_all_arr          float64
nb_delayed_15              float64
avg_delay_15               float64
nb_delayed_30              float64
nb_delayed_60              float64
pct_cause_external         float64
pct_cause_infra            float64
pct_cause_traffic          float64
pct_cause_rolling_stock    float64
pct_cause_station          float64
pct_cause_passenger        float64
dtype: object


## 5. Conversion de la colonne date

La colonne contient cinq formats. On les ramene a deux, puis on convertit.

Premiere etape, uniformiser les separateurs. `.str.replace("/", "-")` et
`.str.replace(" ", "-")` transforment `2018/01` et `2018 01` en `2018-01`.
`.str.strip()` traite le cas `2018-01 ` avec un espace en fin.

Il reste alors deux formats : annee-mois (`2018-01`) et mois-annee (`01-2018`).

Deuxieme etape, `pd.to_datetime()` transforme le texte en vraie date.
L'argument `format="%Y-%m"` signifie annee sur 4 chiffres puis mois, et
`errors="coerce"` met `NaT`, une date vide, quand le format ne correspond pas.
On fait donc un premier essai avec ce format, puis un deuxieme essai avec
`format="%m-%Y"` pour recuperer les valeurs comme `01-2018`.

`.fillna()` remplit les trous du premier essai avec les valeurs du second.
Sans cette double passe, les 77 lignes ecrites en mois-annee seraient perdues.

`.min()` et `.max()` donnent la plus ancienne et la plus recente date, pour
verifier que la periode couverte est plausible.

In [30]:
df["date"] = df["date"].str.strip()
df["date"] = df["date"].str.replace("/", "-", regex=False)
df["date"] = df["date"].str.replace(" ", "-", regex=False)

date_annee_mois = pd.to_datetime(df["date"], format="%Y-%m", errors="coerce")
date_mois_annee = pd.to_datetime(df["date"], format="%m-%Y", errors="coerce")

df["date"] = date_annee_mois.fillna(date_mois_annee)

print("Dates non converties :", df["date"].isna().sum())
print("Periode :", df["date"].min(), "a", df["date"].max())

Dates non converties : 60
Periode : 2018-01-01 00:00:00 a 2025-12-01 00:00:00


## 6. Suppression des doublons

`.drop_duplicates()` supprime les lignes identiques et garde la premiere
occurrence. `len()` donne le nombre de lignes, avant et apres, pour mesurer
combien ont ete retirees.

Cette etape arrive apres le nettoyage du texte et des dates, et non avant : deux
lignes identiques ecrites differemment n'etaient pas reconnues comme des doublons
tant que le texte n'etait pas uniformise. On en trouve ainsi 235 au lieu de 174.

In [31]:
print("Lignes avant :", len(df))

df = df.drop_duplicates()

print("Lignes apres :", len(df))

Lignes avant : 12070
Lignes apres : 11834


## 7. Traitement des valeurs manquantes

Trois traitements differents, selon la nature de la colonne.

### 7.1 Suppression des colonnes de commentaires

Les trois colonnes `cancel_comment`, `dep_comment` et
`arr_comment` contiennent du texte libre et sont vides a environ 90 pour cent.
Les imputer n'aurait aucun sens. `.drop(columns=[...])` les supprime.

In [32]:
df = df.drop(columns=["cancel_comment", "dep_comment", "arr_comment"])

print("Colonnes restantes :", len(df.columns))

Colonnes restantes : 23


### 7.2 Suppression des lignes inexploitables

Une ligne sans date, sans gare ou sans retard
a l'arrivee ne decrit rien. Le retard a l'arrivee est en plus la variable que le
Step 3 devra predire : l'inventer reviendrait a entrainer le modele sur des
valeurs fabriquees et a fausser toutes les metriques.

`.dropna(subset=[...])` supprime les lignes ou l'une de ces colonnes est vide.

In [33]:
df = df.dropna(
    subset=["date", "departure_station", "arrival_station", "avg_delay_all_arr"]
)

print("Lignes restantes :", len(df))

Lignes restantes : 11425


### 7.3 Imputation des trous ponctuels par la mediane

Pour les autres colonnes de nombres, le trou est ponctuel et la ligne reste
utilisable.

`.median()` calcule la valeur du milieu de la colonne et `.fillna()` remplace les
vides par cette valeur. On prend la mediane plutot que la moyenne parce que la
distribution des retards est etiree vers la droite : quelques mois
catastrophiques tirent la moyenne vers le haut, alors que la mediane n'en est pas
affectee.

Pour `service`, qui est une colonne texte, une mediane n'a aucun sens. `.mode()`
donne la valeur la plus frequente et `[0]` en prend la premiere.

Le double `.sum()` a la fin compte d'abord par colonne, puis additionne le tout :
il doit afficher 0.

In [34]:
for col in colonnes_nombres:
    df[col] = df[col].fillna(df[col].median())

df["service"] = df["service"].fillna(df["service"].mode()[0])

print("Valeurs manquantes restantes :", df.isna().sum().sum())

Valeurs manquantes restantes : 0


## 8. Suppression des valeurs impossibles

Le nettoyage precedent a rendu les colonnes exploitables, mais certaines valeurs
restent absurdes. Quatre regles de bon sens les eliminent.

`.between(-30, 300)` met `True` quand la valeur est comprise entre les deux
bornes, et `df[condition]` ne garde que les lignes ou la condition est vraie.
Les bornes viennent du metier : un retard moyen legerement negatif existe, ce
sont des trains en avance, mais `-472` minutes signifierait huit heures d'avance.

Les trois filtres suivants suivent la meme logique : un mois sans train prevu ne
decrit aucun trajet, `0` n'est pas un nom de gare, et il ne peut pas y avoir plus
de trains en retard que de trains prevus.

In [35]:
df = df[df["avg_delay_all_arr"].between(-30, 300)]

df = df[df["nb_scheduled"] > 0]

df = df[df["departure_station"] != "0"]
df = df[df["arrival_station"] != "0"]

df = df[df["nb_delayed_arr"] <= df["nb_scheduled"]]
df = df[df["nb_delayed_dep"] <= df["nb_scheduled"]]

print("Lignes restantes :", len(df))

Lignes restantes : 11278


## 9. Feature engineering

On cree de nouvelles colonnes a partir de celles qui existent deja, pour donner
au modele des informations qu'il ne peut pas deviner seul.

Une precision sur le sujet : il demande des variables comme le jour de la semaine
ou les heures de pointe. Ce n'est pas possible ici. Le dataset est mensuel, une
ligne resume un trajet sur un mois entier, il ne contient ni jour ni horaire. Les
variables temporelles disponibles sont donc l'annee, le mois, le trimestre et la
saison.

### 9.1 Les variables temporelles

`.dt` donne acces aux morceaux d'une date : `.dt.year` pour l'annee, `.dt.month`
pour le numero du mois, `.dt.quarter` pour le trimestre. Cet accesseur ne
fonctionne que parce que la section 5 a converti la colonne en vraie date.

Le dictionnaire `saisons` associe chaque numero de mois a une saison, et `.map()`
remplace chaque numero par la saison correspondante.

`.isin([7, 8, 12])` met `True` si le mois fait partie des mois de vacances, et
`.astype(int)` transforme ces `True` et `False` en 1 et 0, format que comprend un
modele de machine learning.

In [36]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter

saisons = {
    1: "Hiver",
    2: "Hiver",
    12: "Hiver",
    3: "Printemps",
    4: "Printemps",
    5: "Printemps",
    6: "Ete",
    7: "Ete",
    8: "Ete",
    9: "Automne",
    10: "Automne",
    11: "Automne",
}

df["season"] = df["month"].map(saisons)

df["is_holiday_month"] = df["month"].isin([7, 8, 12]).astype(int)

df[["date", "year", "month", "quarter", "season", "is_holiday_month"]].head()

,date,year,month,quarter,season,is_holiday_month
0,2018-01-01,2018,1,1,Hiver,0
1,2018-01-01,2018,1,1,Hiver,0
2,2018-01-01,2018,1,1,Hiver,0
3,2018-01-01,2018,1,1,Hiver,0
4,2018-01-01,2018,1,1,Hiver,0


### 9.2 Identification du trajet et du type de service

L'addition de deux colonnes texte les colle element par element : la gare de
depart, un separateur, la gare d'arrivee. La liaison devient identifiable en une
seule colonne, ce qui servira aux regroupements et aux filtres du dashboard.

La comparaison `df["service"] == "INTERNATIONAL"` produit `True` ou `False`, et
`.astype(int)` la transforme en 1 ou 0.

In [37]:
df["route"] = df["departure_station"] + " - " + df["arrival_station"]

df["is_international"] = (df["service"] == "INTERNATIONAL").astype(int)

print(df["route"].head())

0     BORDEAUX ST JEAN - PARIS MONTPARNASSE
1              LE MANS - PARIS MONTPARNASSE
2    PARIS MONTPARNASSE - LA ROCHELLE VILLE
3               PARIS MONTPARNASSE - NANTES
4             POITIERS - PARIS MONTPARNASSE
Name: route, dtype: str


### 9.3 Les taux d'exploitation

Un effectif brut ne veut rien dire seul : 147 trains en retard sur 870 trains
prevus est un bon mois, 147 sur 200 est une catastrophe. Un modele qui recoit
l'effectif apprend surtout la taille de la liaison, pas sa qualite. Le taux, lui,
est comparable entre toutes les liaisons et tous les mois.

Les divisions et soustractions sont vectorisees : pandas les applique ligne par
ligne sur toute la colonne, sans boucle. `.round(2)` arrondit a deux decimales.

`delay_amplification` est une soustraction entre le retard a l'arrivee et le
retard au depart. Positive, le train a perdu du temps pendant le trajet.
Negative, il en a rattrape. Cette information n'existe nulle part dans le fichier
d'origine, elle nait du calcul.

In [38]:
df["cancellation_rate"] = (df["nb_cancelled"] / df["nb_scheduled"] * 100).round(2)
df["delay_rate_arr"] = (df["nb_delayed_arr"] / df["nb_scheduled"] * 100).round(2)
df["punctuality_rate"] = (100 - df["delay_rate_arr"]).round(2)
df["delay_amplification"] = (df["avg_delay_all_arr"] - df["avg_delay_all_dep"]).round(2)

df[
    ["cancellation_rate", "delay_rate_arr", "punctuality_rate", "delay_amplification"]
].head()

,cancellation_rate,delay_rate_arr,punctuality_rate,delay_amplification
0,0.57,16.90,83.10,2.82
1,0.25,25.86,74.14,0.80
2,0.00,8.41,91.59,2.65
3,0.59,11.42,88.58,4.31
4,0.85,18.86,81.14,1.65


### 9.4 Les categories de retard et de duree

`pd.cut()` decoupe une colonne de nombres en classes. L'argument `bins` donne les
bornes et `labels` le nom de chaque classe. Il y a cinq bornes pour quatre
categories, chaque classe etant l'intervalle entre deux bornes.

`-np.inf` et `np.inf` signifient moins l'infini et plus l'infini. Ils garantissent
qu'aucune valeur ne tombe en dehors des classes et ne devienne vide.

C'est le point du sujet qui demande des categories de retard. Ces colonnes
serviront aussi de filtres dans le dashboard.

`.value_counts()` compte combien de lignes il y a dans chaque categorie.

In [39]:
df["delay_category"] = pd.cut(
    df["avg_delay_all_arr"],
    bins=[-np.inf, 5, 10, 20, np.inf],
    labels=["Faible", "Modere", "Important", "Critique"],
)

df["journey_length"] = pd.cut(
    df["journey_time"],
    bins=[-np.inf, 90, 180, np.inf],
    labels=["Court", "Moyen", "Long"],
)

print(df["delay_category"].value_counts())
print(df["journey_length"].value_counts())

delay_category
Faible       5099
Modere       4520
Important    1538
Critique      121
Name: count, dtype: int64
journey_length
Moyen    4987
Long     4463
Court    1828
Name: count, dtype: int64


## 10. Verification finale

Trois controles avant l'export. C'est ce qu'un correcteur regarde en premier :
la taille du tableau final, le nombre de valeurs manquantes qui doit etre nul, et
le nombre de doublons qui doit l'etre aussi.

In [40]:
print("Taille finale :", df.shape)
print("Valeurs manquantes :", df.isna().sum().sum())
print("Lignes en double :", df.duplicated().sum())

Taille finale : (11278, 36)
Valeurs manquantes : 0
Lignes en double : 0


## 11. Export

`.to_csv()` enregistre le tableau dans un fichier CSV.

`index=False` empeche l'ecriture d'une colonne supplementaire contenant les
numeros de lignes, qui reviendrait sous le nom `Unnamed: 0` au prochain
chargement.

In [41]:
df.to_csv("../cleaned_dataset.csv", index=False)

print("Fichier cleaned_dataset.csv cree")

Fichier cleaned_dataset.csv cree


### 11.1 Relecture de controle

On relit le fichier ecrit pour verifier qu'il est valide et exploitable par le
Step 2.

In [42]:
verification = pd.read_csv("../cleaned_dataset.csv")

print("Taille du fichier relu :", verification.shape)
verification.head()

Taille du fichier relu : (11278, 36)


,date,service,departure_station,arrival_station,journey_time,nb_scheduled,nb_cancelled,nb_delayed_dep,avg_delay_late_dep,avg_delay_all_dep,...,season,is_holiday_month,route,is_international,cancellation_rate,delay_rate_arr,punctuality_rate,delay_amplification,delay_category,journey_length
0,2018-01-01,NATIONAL,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141.00,870.0,5.0,289.0,11.247809,3.693179,...,Hiver,0,BORDEAUX ST JEAN - PARIS MONTPARNASSE,0,0.57,16.90,83.10,2.82,Modere,Moyen
1,2018-01-01,NATIONAL,LE MANS,PARIS MONTPARNASSE,56.00,406.0,1.0,213.0,8.479969,4.567119,...,Hiver,0,LE MANS - PARIS MONTPARNASSE,0,0.25,25.86,74.14,0.80,Modere,Court
2,2018-01-01,NATIONAL,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166.00,226.0,0.0,21.0,6.239683,0.286283,...,Hiver,0,PARIS MONTPARNASSE - LA ROCHELLE VILLE,0,0.00,8.41,91.59,2.65,Faible,Moyen
3,2018-01-01,NATIONAL,PARIS MONTPARNASSE,NANTES,216.21,508.0,3.0,71.0,7.235211,0.980000,...,Hiver,0,PARIS MONTPARNASSE - NANTES,0,0.59,11.42,88.58,4.31,Modere,Long
4,2018-01-01,NATIONAL,POITIERS,PARIS MONTPARNASSE,94.00,472.0,4.0,224.0,6.784673,3.229701,...,Hiver,0,POITIERS - PARIS MONTPARNASSE,0,0.85,18.86,81.14,1.65,Faible,Moyen
